In [ ]:
import numpy as np
from collections import Counter
from treenode import TreeNode

class DecisionTreeFromScratch:
    """
    Decision Tree Classifier implemented from scratch
    For training, all possible percentiles of the feature values are used.
    The quality evaluation is done using entropy as a measure.
    """
    def __init__(self, max_depth, min_leaf) -> None:
        """
        Initialize DecisionTree with specified hyperparameters.
        max_depth - maximum depth of the tree.
        min_leaf - minimum number of data points required in a leaf node.
        """
        self.max_depth = max_depth
        self.min_leaf = min_leaf
        self.tree = None
    
    def _entropy(self, class_probabilities) -> float:
        """Calculates entropy given a list of class probabilities."""
        return sum([-p * np.log2(p) for p in class_probabilities if p > 0])
    
    def _class_probabilities(self, labels) -> list:
        """Returns class probabilities based on label distribution."""
        total_count = len(labels)
        return [count / total_count for count in Counter(labels).values()]
    
    def _data_entropy(self, labels) -> float:
        """Calculates entropy for a given set of labels."""
        return self._entropy(self._class_probabilities(labels))
    
    def _partition_entropy(self, subsets) -> float:
        """Calculates weighted entropy for a given partition of data subsets."""
        total_count = sum(len(subset) for subset in subsets)
        return sum(self._data_entropy(subset) * (len(subset) / total_count) for subset in subsets)
    
    def _split(self, data, feature_index, feature_val) -> tuple:
        """Splits the data into two groups based on a threshold feature value."""
        mask = data[:, feature_index] < feature_val
        group1 = data[mask]
        group2 = data[~mask]
        return group1, group2
    
    def _find_best_split(self, data) -> tuple:
        """
        Finds the best feature and threshold to split data by calculating entropy.
        Returns the split with the lowest entropy.
        """
        min_part_entropy = float('inf')
        best_split = None
        
        n_features = data.shape[1] - 1  # Exclude label column
        for idx in range(n_features):
            feature_vals = np.percentile(data[:, idx], q=np.arange(1, 100, 5))
            for feature_val in feature_vals:
                g1, g2 = self._split(data, idx, feature_val)
                part_entropy = self._partition_entropy([g1[:, -1], g2[:, -1]])
                if part_entropy < min_part_entropy:
                    min_part_entropy = part_entropy
                    best_split = (g1, g2, idx, feature_val, min_part_entropy)
        
        return best_split if best_split else (None, None, None, None, None)
    
    def _find_label_probs(self, data) -> np.array:
        """Calculates label probabilities for a given dataset."""
        labels = data[:, -1].astype(int)
        total_labels = len(labels)
        label_probabilities = np.zeros(len(self.labels_in_train), dtype=float)
        for i, label in enumerate(self.labels_in_train):
            count = np.sum(labels == label)
            label_probabilities[i] = count / total_labels
        return label_probabilities
    
    def _create_tree(self, data, current_depth) -> TreeNode:
        """Recursively builds the decision tree."""
        if current_depth > self.max_depth:
            return None
        
        split = self._find_best_split(data)
        if not split:
            return None
        
        split1, split2, split_feature_idx, split_feature_val, _ = split
        label_probabilities = self._find_label_probs(data)
        
        node = TreeNode(data, split_feature_idx, split_feature_val, label_probabilities)
        
        # Check leaf node condition
        if len(split1) < self.min_leaf or len(split2) < self.min_leaf:
            return node
        
        current_depth += 1
        node.left = self._create_tree(split1, current_depth)
        node.right = self._create_tree(split2, current_depth)
        
        return node
    
    def _predict_one_sample(self, X) -> np.array:
        """Predicts the probability distribution for a single sample."""
        node = self.tree
        while node.left or node.right:
            if X[node.feature_idx] < node.feature_val:
                node = node.left
            else:
                node = node.right
        return node.prediction_probs
    
    def train(self, X_train, Y_train) -> None:
        """Trains the decision tree model."""
        self.labels_in_train = np.unique(Y_train)
        train_data = np.concatenate((X_train, np.reshape(Y_train, (-1, 1))), axis=1)
        self.tree = self._create_tree(data=train_data, current_depth=0)
    
    def predict_proba(self, X_set) -> np.array:
        """Returns predicted probabilities for each sample in the dataset."""
        return np.apply_along_axis(self._predict_one_sample, 1, X_set)
    
    def predict(self, X_set) -> np.array:
        """Predicts the class labels for a given dataset."""
        pred_probs = self.predict_proba(X_set)
        return np.argmax(pred_probs, axis=1)
    
    def _print_recursive(self, node, level=0) -> None:
        """Recursively prints the tree structure."""
        if node:
            self._print_recursive(node.left, level + 1)
            print('    ' * level + f"-> Feature: {node.feature_idx}, Threshold: {node.feature_val}, Probabilities: {node.prediction_probs}")
            self._print_recursive(node.right, level + 1)
    
    def print_tree(self) -> None:
        """Prints the entire tree structure."""
        self._print_recursive(node=self.tree)
